# AutoResearch FetchSlide — Colab T4 GPU

Run the from-scratch FetchSlide RL loop on a Colab T4 GPU. Same code as the local `autoresearch` package; only the device changes (`cuda`).

1. Run all cells in order.
2. Cell 2 clones the repo.
3. Cell 3 runs the reference-recipe training (pure sparse + TD3 + reference cadence) on T4.
4. Cell 4 runs the multi-agent autoresearch loop.

All checkpoints and summaries are written under `AUTORESEARCH_RUNS` (default `/content/autoresearch-runs`).

In [ ]:
!pip -q install gymnasium gymnasium-robotics torch 2>&1 | tail -3

In [ ]:
import os, sys, shutil, subprocess

REPO = '/content/autoresearch'
RUNS = os.environ.get('AUTORESEARCH_RUNS', '/content/autoresearch-runs')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/DeconvFFT/fetch-and-slide-HRE-PRE.git', REPO], check=True)
os.makedirs(RUNS, exist_ok=True)
os.environ['AUTORESEARCH_RUNS'] = RUNS
sys.path.insert(0, REPO)
print('REPO', REPO, 'RUNS', RUNS)

## Reference-recipe training (pure sparse + TD3 + reference cadence)

This matches the reference HER+DDPG recipe that hit 80%: pure sparse reward, TD3 (better than DDPG per literature), and the reference cadence (rollouts_per_cycle=2, optimsteps=40). Runs on T4 GPU.

In [ ]:
import json
cfg = {
    'train_episodes': 10000, 'horizon': 50, 'eval_episodes': 50, 'batch_size': 256,
    'warmup_steps': 100, 'updates_per_step': 1, 'eval_every': 1000, 'log_every': 500,
    'device': 'cuda', 'policy_noise': 0.2, 'noise_clip': 0.5, 'actor_delay': 2,
    'her_future': 4, 'her_ratio': 0.8, 'dense_reward': False, 'success_bonus': 0.0,
    'reach_coef': 0.0, 'reach_contact_bonus': 0.0, 'push_coef': 0.0, 'goal_bonus': 0.0,
    'goal_bonus_radius': 0.4, 'actor_l2': 1.0, 'scripted_rollouts': 0, 'scripted_every': 0,
    'rollouts_per_cycle': 2, 'optimsteps': 40,
}
cfg_path = f'{RUNS}/ref_recipe.json'
os.makedirs(RUNS, exist_ok=True)
with open(cfg_path, 'w') as f:
    json.dump(cfg, f)
print('config written:', cfg_path)
print('total steps:', cfg['train_episodes'] * cfg['horizon'])

In [ ]:
import subprocess, sys, os
cfg_path = f'{RUNS}/ref_recipe.json'
out = f'{RUNS}/ref_recipe'
env = dict(os.environ); env['PYTHONPATH'] = REPO + os.pathsep + env.get('PYTHONPATH', '')
r = subprocess.run([sys.executable, '-m', 'autoresearch.worker', '--config', cfg_path, '--output', out],
                   cwd=REPO, env=env, capture_output=True, text=True)
print(r.stdout[-3000:])
print('STDERR tail:', r.stderr[-1000:])

In [ ]:
import json
try:
    m = json.load(open(f'{RUNS}/ref_recipe/metrics.json'))
    print('score:', round(m['score'], 4))
    print('success:', m['metrics']['success_rate'])
    print('dist:', round(m['metrics']['mean_final_distance'], 4))
    print('contact_rate:', m.get('contact_rate'))
except Exception as e:
    print('no metrics yet:', e)

## Multi-agent autoresearch loop

Runs the Karpathy-style autonomous loop with the two-model proposal system (pro strategist + flash implementor) on T4. Requires an OpenRouter API key.

In [ ]:
import os
os.environ['OPENROUTER_API_KEY'] = 'YOUR_OPENROUTER_KEY_HERE'  # <-- set your key
print('key set:', bool(os.environ['OPENROUTER_API_KEY'] and os.environ['OPENROUTER_API_KEY'] != 'YOUR_OPENROUTER_KEY_HERE'))

In [ ]:
import subprocess, sys, os
env = dict(os.environ); env['PYTHONPATH'] = REPO + os.pathsep + env.get('PYTHONPATH', '')
# Run a bounded multi-agent loop (e.g. 5 iterations)
r = subprocess.run([sys.executable, '-m', 'autoresearch.agent_loop', '--tag', 'colab1', '--iterations', '5'],
                   cwd=REPO, env=env, capture_output=True, text=True, timeout=3600)
print(r.stdout[-4000:])
print('STDERR tail:', r.stderr[-1500:])

In [ ]:
import glob
best = sorted(glob.glob(f'{RUNS}/run-*/best_checkpoint.pt'))[-1] if glob.glob(f'{RUNS}/run-*/best_checkpoint.pt') else None
print('best checkpoint:', best)
print('results.tsv:')
!cd {REPO} && cat results.tsv 2>/dev/null | tail -20